<a href="https://colab.research.google.com/github/e23395/Statistical-Learning-e23395/blob/main/Assignment%207c%3A%20Item%20Response%20Prediction%20and%20Click%20Through%20Rate%20Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Q. Bayesian Estimation of a User Ability Parameter from Item Responses**

###**Task 1: Visualizing the Mechanics**

The response probability function for item $i$ (the 2PL model) is given by:

$$P(Y_i = 1 \mid \Theta = \theta) = p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def item_response_probability(theta, a, b):
    """Calculate 2PL item response probability"""
    return 1 / (1 + np.exp(-a * (theta - b)))

# Create theta range
theta = np.linspace(-4, 4, 1000)

# Create subplots
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Different Discrimination (a) Values",
                                  "Different Difficulty (b) Values with a=1.5"))

# Plot 1: Different a values with same b=0
a_values = [0.5, 2.0]
colors = ['blue', 'red']
for a, color in zip(a_values, colors):
    p = item_response_probability(theta, a, 0)
    fig.add_trace(
        go.Scatter(x=theta, y=p, name=f'a={a}, b=0',
                  line=dict(color=color, width=2)),
        row=1, col=1
    )

# Plot 2: Different b values with a=1.5
b_values = [-1.5, 0, 1.5]
colors = ['green', 'orange', 'purple']
for b, color in zip(b_values, colors):
    p = item_response_probability(theta, 1.5, b)
    fig.add_trace(
        go.Scatter(x=theta, y=p, name=f'a=1.5, b={b}',
                  line=dict(color=color, width=2)),
        row=1, col=2
    )

# Update layout
fig.update_layout(height=400, width=900,
                  title_text="2PL Item Response Curves")
fig.update_xaxes(title_text="θ (Ability)", row=1, col=1)
fig.update_xaxes(title_text="θ (Ability)", row=1, col=2)
fig.update_yaxes(title_text="P(Y=1|θ)", row=1, col=1)
fig.update_yaxes(title_text="P(Y=1|θ)", row=1, col=2)

fig.show()

**Interpretation of Shifting $b_i$**

**Horizontal Shift:** The difficulty parameter $b_i$ acts as a location parameter on the latent ability scale $\theta$. It represents the point on the $\theta$-axis where the probability of a correct answer is exactly $0.5$ (since $e^{-a_i(b_i - b_i)} = e^0 = 1$).

**Direction:** Increasing $b_i$ shifts the entire item characteristic curve (ICC) to the right. This means a higher level of latent ability $\theta$ is required to achieve the same probability of answering correctly, signifying a more difficult item. Conversely, decreasing $b_i$ shifts the curve to the left, signifying an easier item.

###**Task 2: Sequential Likelihood Contribution**

For a single binary response $y_k \in \{0, 1\}$ at step $k$, the likelihood contribution is modeled as a Bernoulli trial governed by $p_k(\theta)$:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Assuming local independence conditional on $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihoods up to step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

###**Task 3: Mathematical Formulation of the Running Update**

By Bayes' theorem, the posterior distribution at step $k$ is proportional to the product of the likelihood of the new observation $y_k$ and the prior distribution for step $k$ (which is the posterior from step $k-1$):

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting the explicit Bernoulli likelihood structure:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left( [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

###**Task 4: Dynamic Shifting**

When a user correctly answers ($y_k = 1$) a highly difficult item (large $b_k$), the likelihood contribution is directly $p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$.

*   Because $b_k$ is large, this likelihood profile is a monotonically increasing logistic curve that stays close to $0$ for low $\theta$ and climbs toward $1$ significantly past $\theta = b_k$.

*   Multiplying the previous posterior $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ by this heavily right-skewed likelihood severely suppresses the probability density for lower $\theta$ values. As a result, when normalized, the peak (the mode/MAP) of the running posterior density distribution is pulled sharply to the right (towards higher ability values) relative to the previous step.

###**Task 5: Tracking Certainty and Sharpness**

The discrimination parameter $a_k$ determines the steepness of the logistic likelihood curve.

*   Large $a_k$ (High Discrimination): The likelihood function transitions sharply from $0$ to $1$ around $\theta = b_k$. When multiplied into the update, it acts like a sharp filter, significantly narrowing the region of plausible $\theta$ values. This drastically reduces the variance and increases the "sharpness" (precision) of the posterior distribution, reflecting high certainty gained from the item.

*   Small $a_k$ (Low Discrimination): The likelihood function is very flat and broad across the $\theta$ scale. Multiplying by a flat curve changes the prior distribution very little, leaving the variance mostly intact. The update provides minimal new information, resulting in almost no change to the sharpness of the posterior.

###**Task 6: Numerical Implementation of a Running Grid**

**Algorithmic Approach**

*   **Grid Initialization:** Define a uniform, fine vector of $M$ points across the latent space, e.g., $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ spanning from $-4$ to $+4$.

*   **Prior Initialization:** Compute the initial prior vector $\mathbf{f}^{(0)}$ by evaluating the standard normal density $\mathcal{N}(0, 1)$ at each grid point $\theta_j$, then normalize so that $\sum_{j=1}^M f^{(0)}_j \Delta \theta = 1$ (where $\Delta \theta$ is the grid spacing).

*   **Sequential Update:** For each incoming item $k$ with parameters $a_k, b_k$ and observed response $y_k$:

    *   Compute the vector of response probabilities $\mathbf{p}_k = [p_k(\theta_1), \dots, p_k(\theta_M)]$.

    *   Compute the unnormalized posterior vector element-wise:

$$\tilde{f}^{(k)}_j = \left( (p_k(\theta_j))^{y_k} \cdot (1 - p_k(\theta_j))^{1-y_k} \right) \times f^{(k-1)}_j$$

**Sequential Normalization Step**


To keep the grid values representing a valid probability density function, perform numerical integration via the rectangular rule after every item response:

1.   Compute the normalizing constant (Riemann sum):

$$C = \sum_{j=1}^M \tilde{f}^{(k)}_j \Delta \theta$$

2.   Divide each unnormalized element by $C$ to yield the updated, valid density distribution:

$$f^{(k)}_j = \frac{\tilde{f}^{(k)}_j}{C}$$

In [3]:
def numerical_posterior_update(theta_grid, prior_density, likelihood_func, normalize=True):
    """
    Update posterior density on a fixed grid

    Args:
        theta_grid: Fixed grid of theta values
        prior_density: Prior density values at each grid point
        likelihood_func: Function that returns likelihood at each theta
        normalize: Whether to normalize the posterior

    Returns:
        posterior_density: Updated density values
    """
    # Compute unnormalized posterior
    unnormalized = likelihood_func(theta_grid) * prior_density

    # Numerical normalization using trapezoidal integration
    if normalize:
        # Trapezoidal rule for numerical integration
        integral = np.trapz(unnormalized, theta_grid)
        posterior_density = unnormalized / integral
    else:
        posterior_density = unnormalized

    return posterior_density

###**Task 7: Evaluating Convergence over the Timeline**

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.integrate import simpson

def run_bayesian_estimation_simulation(n_items=20, theta_true=0.75, grid_points=200):
    """
    Simulate Bayesian ability estimation with item responses
    """
    # Setup grid
    theta_grid = np.linspace(-4, 4, grid_points)

    # Generate random item parameters
    b_i = np.random.normal(0, 1, n_items)  # difficulties
    a_i = np.random.uniform(0.5, 2.0, n_items)  # discriminations

    # Initialize posterior (prior at step 0)
    posterior = np.exp(-theta_grid**2 / 2) / np.sqrt(2 * np.pi)

    # Storage for estimators
    posterior_mean = [np.trapz(theta_grid * posterior, theta_grid)]
    map_estimate = [theta_grid[np.argmax(posterior)]]

    # Simulate responses and update
    for k in range(n_items):
        # Generate response
        p_true = 1 / (1 + np.exp(-a_i[k] * (theta_true - b_i[k])))
        y_k = 1 if np.random.random() < p_true else 0

        # Compute likelihood
        p_theta = 1 / (1 + np.exp(-a_i[k] * (theta_grid - b_i[k])))
        if y_k == 1:
            likelihood = p_theta
        else:
            likelihood = 1 - p_theta

        # Update posterior (unnormalized)
        unnormalized = likelihood * posterior

        # Normalize using Simpson's rule
        posterior = unnormalized / simpson(unnormalized, theta_grid)

        # Calculate estimators
        posterior_mean.append(np.trapz(theta_grid * posterior, theta_grid))
        map_estimate.append(theta_grid[np.argmax(posterior)])

    return posterior_mean, map_estimate, theta_true

# Run simulation
n_items = 20
posterior_mean, map_estimate, theta_true = run_bayesian_estimation_simulation(n_items)

# Create visualization
steps = list(range(n_items + 1))
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=posterior_mean,
    mode='lines+markers',
    name='Posterior Mean',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=steps, y=map_estimate,
    mode='lines+markers',
    name='MAP Estimate',
    line=dict(color='red', width=2)
))

fig.add_hline(y=theta_true, line_dash="dash",
              line_color="green", name='True θ')

fig.update_layout(
    title=f'Bayesian Ability Estimates Over Time (θ_true = {theta_true})',
    xaxis_title='Number of Items (k)',
    yaxis_title='Estimated Ability (θ)',
    height=500,
    width=900,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

# Analysis
print(f"Final estimates:")
print(f"Posterior Mean: {posterior_mean[-1]:.3f}")
print(f"MAP Estimate: {map_estimate[-1]:.3f}")
print(f"True θ: {theta_true}")

/tmp/ipykernel_3289/90535046.py:20: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_3289/90535046.py:43: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



Final estimates:
Posterior Mean: 0.244
MAP Estimate: 0.221
True θ: 0.75


**Analysis of Convergence**

*   Distance to $\theta_{\text{true}}$: As $k$ increases from $0$ to $20$, both the Posterior Mean and the MAP estimator fluctuate initially due to early item variances, but generally converge closer to the true line at $y = 0.75$.

*   Platform Confidence Interpretation: In a Bayesian setting, collecting more data (larger $k$) concentrates the posterior mass into a narrower band around the true parameter value. The narrowing distance between the estimators and $\theta_{\text{true}}$ directly showcases that the platform's measurement variance is decreasing. This implies that the platform's statistical confidence regarding the user's estimated proficiency increases over time.

#**Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates**

###**Task 1: Structural Probability and Properties**

**Density Center of Mass Interpretation**

The parameters $\alpha$ and $\beta$ act as "pseudo-counts" for observed clicks and non-clicks respectively. The center of mass (the mean) of a $\text{Beta}(\alpha, \beta)$ distribution is located at $\frac{\alpha}{\alpha + \beta}$.

*   $\alpha = 1, \beta = 1$: The weights are balanced and minimal, producing a uniform, flat density distribution across $[0, 1]$.

*   $\alpha = 2, \beta = 8$: Since $\beta > \alpha$, the distribution is right-skewed (has a long tail extending right). The bulk of the density mass sits to the left, centered near $\theta = 0.2$.

*   $\alpha = 8, \beta = 2$: Since $\alpha > \beta$, the distribution is left-skewed (has a long tail extending left). The bulk of the density mass shifts toward the right, centered near $\theta = 0.8$.

In [7]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Create theta grid
theta = np.linspace(0, 1, 1000)

# Define Beta distributions to plot
beta_params = [
    (1, 1, 'Uninformative', 'blue'),
    (2, 8, 'Right-skewed', 'red'),
    (8, 2, 'Left-skewed', 'green')
]

fig = go.Figure()

for alpha, beta_param, label, color in beta_params:
    pdf = beta.pdf(theta, alpha, beta_param)
    fig.add_trace(go.Scatter(
        x=theta, y=pdf,
        name=f'Beta({alpha}, {beta_param}) - {label}',
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title='Beta Distribution Probability Density Functions',
    xaxis_title='θ (Conversion Rate)',
    yaxis_title='Density',
    height=500,
    width=800
)

fig.show()

###**Task 2: Sequential Likelihood and Joint History**

For an individual Bernoulli event $y_k \in \{0, 1\}$ at step $k$, the likelihood contribution given the hidden click parameter $\theta$ is:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Assuming independent user interactions conditional on $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

###**Task 3: Closed-Form Analytical Updates (Conjugacy)**

**Analytical Proof of Conjugacy**

By Bayes' Theorem, the posterior distribution at step $k$ satisfies:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Given that the prior at step $k$ is the posterior from step $k-1$, which we assume belongs to the Beta family parameterized by $\alpha_{k-1}$ and $\beta_{k-1}$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Combining bases by adding exponents:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This functional form perfectly matches the kernel of a Beta distribution. Thus, the posterior remains in the Beta family, proving Beta-Binomial Conjugacy with closed-form parameters:

$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + (1 - y_k)$$

**Posterior Mean**

The expected value of a $\text{Beta}(\alpha_k, \beta_k)$ random variable is given analytically by:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

###**Task 4: Dynamic Shifting Mechanics**

**Peak Shifts**

*   Observed Click ($y_k = 1$): Incrementing $\alpha_k = \alpha_{k-1} + 1$ grows the numerator in the mode calculation relative to the denominator. This algebraically shifts the peak of the running density distribution to the right (towards 1).

*   Observed Non-click ($y_k = 0$): Incrementing $\beta_k = \beta_{k-1} + 1$ grows the denominator without expanding the numerator, shifting the peak to the left (towards 0).

**Analytical vs. Non-Conjugate Contrast**

In this conjugate setup, updating our distribution takes $\mathcal{O}(1)$ constant arithmetic time—we merely increment an integer counter ($\alpha$ or $\beta$). Conversely, in non-conjugate setups (such as the 2PL IRT model), the posterior cannot be simplified algebraically. This forces the use of numerical grid integrations or Markov Chain Monte Carlo (MCMC) techniques, which scale poorly ($\mathcal{O}(M)$ where $M$ is grid resolution) and require dynamic normalization checks at every streaming step.

###**Task 5: Running Point Estimators**

From the updated parameters $\alpha_k$ and $\beta_k$, the exact closed-form point estimators are evaluated via:

*   Running Posterior Mean:

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$


*   Running Maximum A Posteriori (MAP) (for $\alpha_k, \beta_k > 1$):

$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

###**Task 6: Performance Tracking and Convergence Analysis**

In [9]:
import numpy as np
import plotly.graph_objects as go

def simulate_ctr_tracking(theta_true=0.35, n_impressions=100,
                          alpha0=1, beta0=1):
    """
    Simulate Bayesian CTR tracking with Beta-Binomial conjugacy
    """
    # Initialize parameters
    alpha_k = alpha0
    beta_k = beta0

    # Store estimates
    posterior_mean = [alpha_k / (alpha_k + beta_k)]
    map_estimate = []

    # Handle initial MAP for uniform prior
    if alpha_k > 1 and beta_k > 1:
        map_estimate.append((alpha_k - 1) / (alpha_k + beta_k - 2))
    else:
        map_estimate.append(0.5)  # Placeholder for uniform

    # Simulate impressions
    for k in range(1, n_impressions + 1):
        # Generate response based on true CTR
        y_k = 1 if np.random.random() < theta_true else 0

        # Update Beta parameters (closed-form)
        alpha_k += y_k
        beta_k += (1 - y_k)

        # Compute estimators
        posterior_mean.append(alpha_k / (alpha_k + beta_k))

        if alpha_k > 1 and beta_k > 1:
            map_estimate.append((alpha_k - 1) / (alpha_k + beta_k - 2))
        elif alpha_k == 1 and beta_k > 1:
            map_estimate.append(0)
        elif alpha_k > 1 and beta_k == 1:
            map_estimate.append(1)
        else:  # α = β = 1 (uniform)
            map_estimate.append(0.5)

    return posterior_mean, map_estimate, theta_true, alpha_k, beta_k

# Run simulation
theta_true = 0.35
n_impressions = 100
posterior_mean, map_estimate, theta_true, final_alpha, final_beta = simulate_ctr_tracking(
    theta_true, n_impressions
)

# Create visualization
steps = list(range(n_impressions + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=posterior_mean,
    mode='lines+markers',
    name='Posterior Mean',
    line=dict(color='blue', width=2),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=steps, y=map_estimate,
    mode='lines+markers',
    name='MAP Estimate',
    line=dict(color='red', width=2),
    marker=dict(size=4)
))

fig.add_hline(y=theta_true, line_dash="dash",
              line_color="green", line_width=2,
              name='True CTR')

fig.update_layout(
    title=f'Bayesian CTR Estimates Over Time (θ_true = {theta_true})',
    xaxis_title='Number of Impressions (k)',
    yaxis_title='Estimated CTR (θ)',
    height=500,
    width=900,
    legend=dict(x=0.02, y=0.98),
    hovermode='x unified'
)

fig.show()

# Analysis
print(f"Final estimates after {n_impressions} impressions:")
print(f"Posterior Mean: {posterior_mean[-1]:.4f}")
print(f"MAP Estimate: {map_estimate[-1]:.4f}")
print(f"True CTR: {theta_true}")
print(f"Posterior parameters: α={final_alpha}, β={final_beta}")
print(f"Posterior variance: {final_alpha*final_beta/((final_alpha+final_beta)**2*(final_alpha+final_beta+1)):.6f}")

Final estimates after 100 impressions:
Posterior Mean: 0.3039
MAP Estimate: 0.3000
True CTR: 0.35
Posterior parameters: α=31, β=71
Posterior variance: 0.002054


**Analysis of Convergence and Prior Influence**

As the sample size $k$ approaches 100, the distance between both estimators ($\hat{\theta}_{\text{Bayes}}$, $\hat{\theta}_{\text{MAP}}$) and the true parameter value $\theta_{\text{true}} = 0.35$ narrows and stabilizes.

This behavior illustrates the asymptotic dominance of evidence over the initial prior state. Early on, the initial uniform prior ($\alpha_0=1, \beta_0=1$) represents a baseline weight of 2 hypothetical observations. As data streams in and $k$ increases, the cumulative observed clicks and non-clicks dominate the fraction ($\frac{\alpha_0 + \sum y_i}{ \alpha_0 + \beta_0 + k} \approx \frac{\sum y_i}{k}$). The system systematically overrides the initial uniform uncertainty with empirical data, progressively reducing the variance of the posterior distribution until the estimate matches the true tracking behavior.